# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">시작하기에 앞서 잠시 안내드립니다 --</h2>
            <span style="color:#f71;">강의에 필요한 유용한 리소스들이 모여 있는 페이지를 안내해 드리고자 합니다. 이 페이지에는 모든 강의 슬라이드 링크가 포함되어 있습니다.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            이 페이지를 북마크해 두세요. 앞으로도 유용한 링크들을 지속적으로 업데이트할 예정입니다.
            </span>
        </td>
    </tr>
</table>

## 먼저, Chat Completions API에 대해 알아봅시다

1. LLM을 호출하는 가장 간단한 방법입니다.
2. "여기에 대화 내용이 있으니, 다음에 올 내용을 예측해 줘"라고 요청하는 방식이기 때문에 'Chat Completions(채팅 완성)' API라고 불립니다.
3. Chat Completions API는 OpenAI가 처음 만들었지만, 현재는 매우 대중화되어 거의 모든 곳에서 표준처럼 사용하고 있습니다!

### 다시 한번 OpenAI를 호출하는 것으로 시작해 보겠습니다. 하지만 OpenAI를 사용하지 않는 분들도 걱정 마세요. 곧 다른 대안들도 다룰 예정입니다!

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


## '엔드포인트(Endpoint)'가 무엇인지 알고 계신가요?

잘 모르신다면 가이드 폴더에 있는 **Technical Foundations** 가이드를 다시 한번 검토해 주세요.

그리고 여기, 여러분이 흥미를 느낄만한 엔드포인트 하나를 소개합니다...

In [ ]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

In [ ]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

In [ ]:
response.json()["choices"][0]["message"]["content"]

# openai 패키지란 무엇일까요?

이것은 **파이썬 클라이언트 라이브러리(Python Client Library)**로 알려져 있습니다.

이 패키지는 앞서 살펴본 HTTP 엔드포인트 호출 과정을 사용하기 편하게 감싸놓은 **래퍼(Wrapper)**일 뿐입니다.

복잡하고 다루기 힘든 JSON 객체를 직접 만지는 대신, 깔끔한 파이썬 코드로 작업할 수 있게 도와주는 역할을 하죠.

그게 전부입니다. 오픈 소스이며 매우 가볍습니다. 어떤 사람들은 이 패키지 안에 OpenAI의 모델 코드 자체가 들어있다고 생각하기도 하지만, 실제로는 그렇지 않습니다!

In [ ]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



## 그리고 이런 멋진 일이 일어났습니다:
OpenAI의 Chat Completions API가 워낙 인기를 끌다 보니, 다른 모델 제공업체들도 이와 동일한 형태의 엔드포인트를 만들기 시작했습니다. 이를 **"OpenAI 호환 엔드포인트"**라고 부릅니다.

예를 들어, Google도 이런 주소를 만들었습니다:
https://generativelanguage.googleapis.com/v1beta/openai/

OpenAI는 감사하게도 자기들이 만든 파이썬 라이브러리를 다른 제공업체에서도 쓸 수 있게 허용했습니다. 덕분에 우리는 URL과 API 키만 바꿔서 동일한 도구를 사용할 수 있습니다.

코드 예시 (개념 설명용):
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)

코드에 'OpenAI'라는 이름이 들어가지만, 이는 단지 '호출 도구'일 뿐이며 실제 실행은 Google의 모델이 담당합니다. 헷갈린다면 가이드 폴더의 Guide 9를 확인해 보세요.

## (선택 사항) Google Gemini 사용해 보기:
Google AI Studio 방문 및 로그인

API 키 설정 페이지에서 키 발급

.env 파일에 아래 내용을 추가하고 저장하세요:
GOOGLE_API_KEY=여러분의_실제_키_내용

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## Ollama 또한 OpenAI 호환 엔드포인트를 제공합니다

...심지어 여러분의 로컬 컴퓨터에서 작동합니다!

Ollama는 로컬 환경에서 대규모 언어 모델(LLM)을 실행할 수 있게 해주는 도구이며, OpenAI와 동일한 방식의 호출 인터페이스를 제공합니다.



**주의 사항:**
만약 다음 셀을 실행했을 때 **"Ollama is running"**이라는 문구가 출력되지 않는다면, 터미널(Terminal)을 열고 아래 명령어를 입력하여 Ollama 서비스를 먼저 실행해 주세요:

`ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### Meta의 llama3.2 다운로드하기

만약 컴퓨터 사양이 낮거나 성능이 부족하다면 `llama3.2:1b`로 변경하여 사용하세요.

**주의:** `llama3.3`이나 `llama4`는 사용하지 마세요! 일반적인 개인용 컴퓨터에서 돌리기에는 모델 크기가 너무 큽니다.

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# 숙제 과제 (HOMEWORK EXERCISE)

1일 차 프로젝트였던 '웹페이지 요약하기'를 업그레이드하여, OpenAI 대신 **Ollama를 통해 로컬에서 실행되는 오픈 소스 모델**을 사용하도록 수정하세요.

유료 API를 사용하고 싶지 않다면, 앞으로 진행될 모든 프로젝트에 이 방식을 적용할 수 있습니다.

**장점:**
1. API 비용 없음 - 오픈 소스 모델 사용
2. 데이터가 외부로 유출되지 않음 (내 컴퓨터 안에서만 처리)

**단점:**
1. 프론티어 모델(GPT-4 등)에 비해 성능이 현저히 낮음

---

## Ollama 설치 방법 복습

단순히 [ollama.com](https://ollama.com)에 접속해서 설치하면 됩니다!

설치가 완료되면 Ollama 서버가 로컬에서 이미 실행 중이어야 합니다.
[http://localhost:11434/](http://localhost:11434/) 주소를 방문해 보세요.

`Ollama is running`이라는 메시지가 보인다면 정상입니다.

만약 보이지 않는다면:
1. 새로운 터미널(Mac) 또는 Powershell(Windows)을 열고 `ollama serve`를 입력합니다.
2. 또 다른 터미널/Powershell 창을 열고 `ollama pull llama3.2`를 입력합니다.
3. 다시 [http://localhost:11434/](http://localhost:11434/)에 접속해서 확인해 보세요.

Ollama가 너무 느리다면 대안으로 `llama3.2:1b`를 사용해 보세요. 터미널에서 `ollama pull llama3.2:1b`를 실행한 후, 코드에서 `MODEL = "llama3.2"` 부분을 `MODEL = "llama3.2:1b"`로 수정하면 됩니다.